## Cell 1: Setup
**What this demonstrates**: Install all dependencies, load API keys, and initialise the three models used across the Ensemble RAG pipeline -- Claude Haiku for cheap HyDE hypothesis generation, Claude Sonnet for final answer generation, and OpenAI embeddings for the shared vector space.
**Expected output**: OK Setup complete with all model names, ensemble weight vector, and masked key suffixes.

In [ ]:
# -- Install dependencies --------------------------------------------------
# rank-bm25: BM25 index for the Hybrid RAG strategy inside the ensemble
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv rank-bm25 anthropic

# -- Standard library -------------------------------------------------------
import os
import time
import pathlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any

# -- Third-party ------------------------------------------------------------
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

# -- Load API keys ----------------------------------------------------------
load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set -- add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set -- add it to .env'

# -- Constants --------------------------------------------------------------
# Haiku for HyDE hypothesis: only needs to match document vocabulary, not be accurate
HYDE_MODEL   = 'claude-haiku-4-5-20251001'
# Sonnet for final answer: reasoning quality matters here, retrieval is already done
ANSWER_MODEL = 'claude-sonnet-4-6'
EMBED_MODEL  = 'text-embedding-3-small'
CHROMA_DIR   = './chroma_ensemble_rag'
CHUNK_SIZE   = 450
TOP_N        = 15   # candidates per retriever before ensemble fusion
TOP_K        = 5    # final chunks after weighted RRF fusion
# Hybrid gets highest weight: it is itself a two-strategy fusion (BM25 + dense).
# Naive (pure dense) is the reliable semantic baseline.
# HyDE adds vocabulary bridging but introduces hypothesis noise -- lowest weight.
ENSEMBLE_WEIGHTS: list[float] = [0.35, 0.40, 0.25]  # [Naive, Hybrid, HyDE]

# -- Client initialisation --------------------------------------------------
client:     Anthropic        = Anthropic()
embeddings: OpenAIEmbeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('OK Setup complete')
print(f'  HyDE model:    {HYDE_MODEL}')
print(f'  Answer model:  {ANSWER_MODEL}')
print(f'  Embed model:   {EMBED_MODEL}')
print(f'  Anthropic key: ...{_anthropic_key[-4:]}')
print(f'  OpenAI key:    ...{_openai_key[-4:]}')
print()
print('Ensemble configuration:')
print('  Strategies:  Naive (dense-only) | Hybrid (BM25+dense RRF) | HyDE (hypothetical embed)')
print(f'  Weights:     {ENSEMBLE_WEIGHTS[0]} (Naive)  {ENSEMBLE_WEIGHTS[1]} (Hybrid)  {ENSEMBLE_WEIGHTS[2]} (HyDE)')
print(f'  Candidates:  top-{TOP_N} per strategy -> weighted RRF -> top-{TOP_K} final chunks')
print('  Parallelism: ThreadPoolExecutor (wall time = slowest strategy, not sum)')

## Cell 2: Data -- All Four Sample Documents
**What this demonstrates**: Load all four synthetic fintech documents into a single shared corpus, chunk them uniformly, and build one BM25 index and one Chroma vector store that all three retrieval strategies query. Using all four documents creates a heterogeneous corpus -- exactly the scenario where no single retrieval strategy wins.
**Expected output**: Per-document chunk counts, BM25 and Chroma index sizes, and an explanation of why each document tests a different retrieval regime.

In [ ]:
# -- Locate sample data ----------------------------------------------------
_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _candidates if p.exists()), None)
assert BASE_DIR is not None, (
    'Cannot find shared/sample_data/. '
    'Run from the repo root or from modules/09_ensemble_rag/.'
)

# Load all four documents -- each represents a different fintech document type
DOCS: dict[str, str] = {
    'Basel III Capital Requirements': (BASE_DIR / 'basel_iii_excerpt.txt').read_text(encoding='utf-8'),
    'Meridian Bank Lending Policy':   (BASE_DIR / 'fintech_policy.txt').read_text(encoding='utf-8'),
    'ISDA Master Agreement':          (BASE_DIR / 'isda_excerpt.txt').read_text(encoding='utf-8'),
    'Pinnacle Financial Q3 Earnings': (BASE_DIR / 'earnings_report.txt').read_text(encoding='utf-8'),
}

# -- Chunk all documents into a single flat corpus --------------------------
# Flat corpus is essential: all three retrieval strategies query the same chunks.
# chunk_meta[i] records which document chunk i came from, for source attribution.
splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n', '\n', '. ', ' '],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=60,
)

all_chunks: list[str] = []
chunk_meta: list[dict] = []

for doc_name, doc_text in DOCS.items():
    for i, chunk in enumerate(splitter.split_text(doc_text)):
        all_chunks.append(chunk)
        chunk_meta.append({'source': doc_name, 'local_idx': i})

# -- Build BM25 index -------------------------------------------------------
# Tokenize consistently: lowercase whitespace split, no stopword removal.
# Financial terms like 'margin', 'rate', 'under' carry legal meaning -- keep them.
def tokenize(text: str) -> list[str]:
    return text.lower().split()

tokenized_corpus = [tokenize(c) for c in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)

# -- Build dense (Chroma) index ---------------------------------------------
# chunk_idx in metadata bridges the two index spaces: BM25 and Chroma share the same key.
lc_docs = [
    Document(page_content=chunk, metadata={**chunk_meta[i], 'chunk_idx': i})
    for i, chunk in enumerate(all_chunks)
]
vectorstore = Chroma.from_documents(
    documents=lc_docs,
    embedding=embeddings,
    collection_name='ensemble_rag_corpus',
    persist_directory=CHROMA_DIR,
    collection_metadata={'hnsw:space': 'cosine'},
)

# -- Print corpus stats -----------------------------------------------------
print(f'Corpus: {len(DOCS)} documents  ->  {len(all_chunks)} chunks total')
print()
for doc_name in DOCS:
    n_chunks   = sum(1 for m in chunk_meta if m['source'] == doc_name)
    char_count = len(DOCS[doc_name])
    print(f'  {doc_name:42s}  {n_chunks:3d} chunks  ({char_count:,} chars)')
print()
print(f'BM25 index:  {len(all_chunks)} entries')
print(f'Chroma:      {vectorstore._collection.count()} vectors')
print()
print('Why all four documents?')
print('  Each tests a different retrieval regime -- no single strategy wins across all:')
print()
print('  Basel III (regulatory)  -> exact numeric thresholds (4.5%, 6%, 8%) and defined terms')
print('                             BM25 excels at exact regulatory term lookup')
print()
print('  Lending Policy          -> procedural cross-references between sections')
print('                             Dense excels at semantic paraphrase retrieval')
print()
print('  ISDA Agreement (legal)  -> nested clause identifiers (Section 5(a)(i))')
print('                             BM25 mandatory for exact legal citation lookup')
print()
print('  Earnings Report (mixed) -> KPI tables + analytical narrative')
print('                             HyDE bridges abstract question to report vocabulary')

## Cell 3: Core -- Three Retrieval Strategies + Weighted RRF Ensemble
**What this demonstrates**: Implement three independent retrieval strategies (Naive dense, Hybrid BM25+dense, HyDE), a weighted 3-way Reciprocal Rank Fusion combiner, and a parallel ensemble runner using `ThreadPoolExecutor`. Each strategy is independently callable for comparison in Cell 5.
**Expected output**: Strategy and combiner function definitions confirmed, with the ensemble weight vector and parallelism model summarised.

In [ ]:
# ===========================================================================
# Strategy 1 -- Naive RAG: pure dense cosine similarity
# ===========================================================================
def naive_retrieve(query: str, n: int = TOP_N) -> list[tuple[int, float]]:
    # Retrieve top-n chunks by cosine similarity to the query embedding.
    # Strength: handles semantic paraphrase well.
    # Weakness: misses exact term matches (rule numbers, legal citations).
    # similarity_search_with_score returns (Document, cosine_distance), distance in [0,2].
    # Rescale to similarity in [0,1] for comparable display output.
    raw = vectorstore.similarity_search_with_score(query, k=n)
    return [(int(doc.metadata['chunk_idx']), 1.0 - dist / 2.0) for doc, dist in raw]


# ===========================================================================
# Strategy 2 -- Hybrid RAG: BM25 + dense via Reciprocal Rank Fusion
# ===========================================================================
def _bm25_retrieve(query: str, n: int) -> list[tuple[int, float]]:
    # BM25 Okapi retrieval -- internal helper used by hybrid_retrieve.
    scores = bm25.get_scores(tokenize(query))
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n]
    # Drop zero-score chunks -- no token overlap; they would dilute RRF.
    return [(i, float(scores[i])) for i in top if scores[i] > 0]


def _two_way_rrf(
    ranked_a: list[tuple[int, float]],
    ranked_b: list[tuple[int, float]],
    k: int = 60,
) -> list[tuple[int, float]]:
    # Standard 2-way RRF: merge two ranked lists by rank position, not raw score.
    # RRF score = 1/(k+rank_a) + 1/(k+rank_b); k=60 is the Ma et al. default.
    rrf: dict[int, float] = {}
    for rank, (idx, _) in enumerate(ranked_a, start=1):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (k + rank)
    for rank, (idx, _) in enumerate(ranked_b, start=1):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(rrf.items(), key=lambda x: x[1], reverse=True)


def hybrid_retrieve(query: str, n: int = TOP_N) -> list[tuple[int, float]]:
    # BM25 + dense retrieval fused via Reciprocal Rank Fusion.
    # Strength: covers keyword-exact and semantic queries in one pass.
    # Weakness: still limited to what either BM25 or dense can surface individually.
    bm25_res  = _bm25_retrieve(query, n)
    dense_res = naive_retrieve(query, n)
    return _two_way_rrf(bm25_res, dense_res)[:n]


# ===========================================================================
# Strategy 3 -- HyDE: Hypothetical Document Embedding
# ===========================================================================
def _generate_hypothesis(query: str) -> str:
    # Ask Haiku to write a document-style passage answering the query.
    # The hypothesis only needs vocabulary match with the corpus, not factual accuracy.
    # A hedged or short hypothesis embeds into the wrong region of vector space.
    response = client.messages.create(
        model=HYDE_MODEL,
        max_tokens=180,
        system=(
            'Generate a 120-word hypothetical document excerpt that answers the question. '
            'Write as if from an authoritative financial regulation or policy document: '
            'formal, declarative prose with specific numeric thresholds and defined terms. '
            'Do NOT hedge or use phrases like "typically" or "may vary". '
            'This text will be embedded as a retrieval search vector -- never shown to users.'
        ),
        messages=[{'role': 'user', 'content': f'Question: {query}'}],
    )
    return response.content[0].text.strip()


def hyde_retrieve(query: str, n: int = TOP_N) -> list[tuple[int, float]]:
    # HyDE: generate hypothetical answer -> embed it -> retrieve real chunks.
    # Strength: bridges vocabulary gap between abstract questions and document prose.
    # Weakness: hypothesis generation adds ~500 ms latency; noisy if LLM uses
    #           vocabulary absent from the corpus.
    # The hypothesis is discarded after retrieval -- never included in the answer prompt.
    hypothesis = _generate_hypothesis(query)
    hyp_vector = embeddings.embed_query(hypothesis)
    # Search using hypothetical vector, NOT the original query embedding.
    raw = vectorstore.similarity_search_by_vector(hyp_vector, k=n)
    # similarity_search_by_vector does not return distances; assign descending rank proxy.
    # 1.0 for rank 1, 0.95 for rank 2, etc. -- enough for weighted RRF participation.
    return [(int(doc.metadata['chunk_idx']), 1.0 - i * 0.05) for i, doc in enumerate(raw)]


# ===========================================================================
# Weighted 3-way RRF Combiner
# ===========================================================================
def weighted_rrf_ensemble(
    ranked_lists:   list[list[tuple[int, float]]],
    weights:        list[float],
    strategy_names: list[str],
    k:              int = 60,
    top_k:          int = TOP_K,
) -> list[dict[str, Any]]:
    # Merge N ranked lists via weighted Reciprocal Rank Fusion.
    #
    # Score for chunk c = sum_i  weight_i / (k + rank_i(c))
    #
    # A chunk appearing in multiple lists earns contributions from each retriever.
    # Cross-strategy agreement is the reward signal: a chunk ranked 10th by two
    # strategies beats a chunk ranked 1st by only one (with equal weights).
    # Chunks absent from a list are not penalised -- nothing is discarded.
    assert len(ranked_lists) == len(weights) == len(strategy_names)

    rrf_scores: dict[int, float]     = {}
    found_in:   dict[int, list[str]] = {}

    for name, ranked, weight in zip(strategy_names, ranked_lists, weights):
        for rank, (chunk_idx, _) in enumerate(ranked, start=1):
            rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0.0) + weight / (k + rank)
            found_in.setdefault(chunk_idx, []).append(name)

    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [
        {
            'chunk_idx': idx,
            'text':      all_chunks[idx],
            'source':    chunk_meta[idx]['source'],
            'rrf_score': score,
            'found_in':  found_in.get(idx, []),
        }
        for idx, score in sorted_results
    ]


# ===========================================================================
# Parallel Ensemble Runner
# ===========================================================================
STRATEGY_NAMES = ['Naive', 'Hybrid', 'HyDE']


def ensemble_retrieve(
    query:   str,
    n:       int         = TOP_N,
    top_k:   int         = TOP_K,
    weights: list[float] = ENSEMBLE_WEIGHTS,
) -> tuple[list[dict[str, Any]], dict[str, float]]:
    # Run all three retrieval strategies in parallel, merge via weighted RRF.
    # ThreadPoolExecutor: wall-clock latency = slowest strategy (HyDE, due to
    # hypothesis LLM call), rather than the sequential sum of all three.
    latencies:   dict[str, float]                   = {}
    raw_results: dict[str, list[tuple[int, float]]] = {}

    def timed(name: str, fn, *args):
        t0 = time.perf_counter()
        result = fn(*args)
        latencies[name] = (time.perf_counter() - t0) * 1000
        return name, result

    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {
            executor.submit(timed, 'Naive',  naive_retrieve,  query, n): 'Naive',
            executor.submit(timed, 'Hybrid', hybrid_retrieve, query, n): 'Hybrid',
            executor.submit(timed, 'HyDE',   hyde_retrieve,   query, n): 'HyDE',
        }
        for future in as_completed(futures):
            name, result = future.result()
            raw_results[name] = result

    ordered = [raw_results[name] for name in STRATEGY_NAMES]
    merged  = weighted_rrf_ensemble(ordered, weights, STRATEGY_NAMES, top_k=top_k)
    return merged, latencies


# -- Answer generation helper -----------------------------------------------
def generate_answer(query: str, chunks: list[dict], system: str) -> str:
    # Generate a grounded answer from a list of retrieved chunk dicts.
    context = '\n\n---\n\n'.join(
        f"[Chunk {i+1} | Source: {c['source']}"
        + (f" | RRF={c['rrf_score']:.5f}" if 'rrf_score' in c else '')
        + (f" | via: {', '.join(c['found_in'])}" if c.get('found_in') else '')
        + f"]\n{c['text']}"
        for i, c in enumerate(chunks)
    )
    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=512,
        system=system,
        messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'}],
    )
    return response.content[0].text


print('Retrieval strategies defined:')
print(f'  Naive  (weight={ENSEMBLE_WEIGHTS[0]}): naive_retrieve(query, n)')
print(f'           -> pure dense cosine similarity')
print(f'  Hybrid (weight={ENSEMBLE_WEIGHTS[1]}): hybrid_retrieve(query, n)')
print(f'           -> BM25 top-{TOP_N} + dense top-{TOP_N} -> 2-way RRF')
print(f'  HyDE   (weight={ENSEMBLE_WEIGHTS[2]}): hyde_retrieve(query, n)')
print(f'           -> hypothetical doc (Haiku) -> embed -> dense search')
print()
print('Combiner: weighted_rrf_ensemble(ranked_lists, weights=[0.35, 0.40, 0.25], k=60)')
print('  Cross-strategy agreement = reward signal; absence from a list = no penalty')
print()
print('Runner: ensemble_retrieve(query)')
print('  ThreadPoolExecutor(max_workers=3) -- wall time = slowest strategy (HyDE)')

## Cell 4: Run -- End-to-End Ensemble Retrieval
**What this demonstrates**: The complete Ensemble RAG pipeline on a capital requirements query that spans multiple documents and requires both exact regulatory term retrieval and semantic understanding of capital adequacy concepts.
**Expected output**: Top-5 ensemble chunks with RRF scores, per-strategy provenance flags, source document attribution, grounded answer with regulatory citations, and per-strategy latency breakdown.

In [ ]:
QUERY = 'What are the key capital requirements and minimum ratio thresholds?'

SYSTEM = (
    'You are a regulatory capital analyst. '
    'Answer using ONLY the provided document excerpts. '
    'Cite the source document and specific section or metric for every threshold. '
    'List all numeric ratios mentioned (percentages, ratios, buffers). '
    'If the context covers multiple regulatory frameworks, distinguish between them.'
)

# -- Run the ensemble pipeline ----------------------------------------------
print(f'Query: {QUERY!r}')
print()
print('Running 3 strategies in parallel...')

t_total = time.perf_counter()
ensemble_chunks, strategy_latencies = ensemble_retrieve(QUERY)
retrieval_ms = (time.perf_counter() - t_total) * 1000

# -- Print retrieved chunks with provenance ---------------------------------
print(f'Ensemble top-{len(ensemble_chunks)} (Naive x{ENSEMBLE_WEIGHTS[0]} + Hybrid x{ENSEMBLE_WEIGHTS[1]} + HyDE x{ENSEMBLE_WEIGHTS[2]} -> weighted RRF):')
print()
for i, r in enumerate(ensemble_chunks, 1):
    consensus = len(r['found_in'])
    # Stars reflect cross-strategy agreement: 3 = found by all 3 strategies
    badge   = '***' if consensus == 3 else '** ' if consensus == 2 else '*  '
    via     = '+'.join(r['found_in'])
    src     = r['source'][:40]
    preview = r['text'][:90].replace('\n', ' ')
    print(f'  [{i}] {badge} RRF={r["rrf_score"]:.5f}  [{via:20s}]')
    print(f'       [{src}]')
    print(f'       {preview!r}...')
    print()

# -- Generate answer from ensemble context ----------------------------------
print('Generating answer from ensemble context...')
t_gen = time.perf_counter()
answer: str = generate_answer(QUERY, ensemble_chunks, SYSTEM)
gen_ms = (time.perf_counter() - t_gen) * 1000

print()
print('=' * 65)
print('ENSEMBLE ANSWER')
print('=' * 65)
print(answer)
print('=' * 65)

# -- Latency breakdown -------------------------------------------------------
print()
print('Latency breakdown:')
for name in STRATEGY_NAMES:
    ms  = strategy_latencies.get(name, 0)
    bar = '|' * int(ms / 50)
    print(f'  {name:6s}  {ms:5.0f} ms  {bar}')
print(f'  Wall:   {retrieval_ms:5.0f} ms  (parallel -- not the sequential sum)')
print(f'  Answer: {gen_ms:5.0f} ms')
print(f'  Total:  {retrieval_ms + gen_ms:5.0f} ms')
print()
print('*** = found by all 3 strategies  ** = found by 2  * = found by 1')
print('Chunks found by all 3 strategies carry the strongest cross-modal signal.')

## Cell 5: Inspect -- All Three Answers, Chunk Coverage, and Combined Answer
**What this demonstrates**: Run each strategy independently to generate its own answer, compare them side-by-side, show which chunks each strategy uniquely contributed to the ensemble, and present the final ensemble answer for direct comparison.
**Expected output**: Three individual answers, a chunk coverage matrix showing which strategies found each ensemble chunk, and the ensemble answer for direct comparison.

In [ ]:
# -- Run each strategy individually and generate its own answer -------------
print(f'Query: {QUERY!r}')
print()
print('Generating one answer per strategy for side-by-side comparison...')
print()

individual_system = (
    'You are a regulatory capital analyst. '
    'Answer using ONLY the provided document excerpts. '
    'Cite the source document for every claim. '
    'List all numeric thresholds mentioned.'
)


def strategy_chunks(name: str, ranked: list[tuple[int, float]], top_k: int = TOP_K) -> list[dict]:
    # Build a chunk-dict list from a raw ranked result (no rrf_score -- single strategy).
    return [
        {'chunk_idx': idx, 'text': all_chunks[idx],
         'source': chunk_meta[idx]['source'], 'found_in': [name]}
        for idx, _ in ranked[:top_k]
    ]


# Run the three retrieval strategies (reuse functions defined in Cell 3)
naive_raw  = naive_retrieve(QUERY)
hybrid_raw = hybrid_retrieve(QUERY)
hyde_raw   = hyde_retrieve(QUERY)

naive_ch  = strategy_chunks('Naive',  naive_raw)
hybrid_ch = strategy_chunks('Hybrid', hybrid_raw)
hyde_ch   = strategy_chunks('HyDE',   hyde_raw)

# Generate all three answers in parallel to save wall-clock time
with ThreadPoolExecutor(max_workers=3) as ex:
    f_n = ex.submit(generate_answer, QUERY, naive_ch,  individual_system)
    f_h = ex.submit(generate_answer, QUERY, hybrid_ch, individual_system)
    f_y = ex.submit(generate_answer, QUERY, hyde_ch,   individual_system)
    naive_answer  = f_n.result()
    hybrid_answer = f_h.result()
    hyde_answer   = f_y.result()

# -- Print all three individual answers -------------------------------------
for label, ans in [
    ('NAIVE RAG  (dense-only)',        naive_answer),
    ('HYBRID RAG (BM25 + dense RRF)',  hybrid_answer),
    ('HyDE RAG  (hypothetical embed)', hyde_answer),
]:
    print('=' * 65)
    print(f'ANSWER -- {label}')
    print('=' * 65)
    print(ans[:550] + ('...' if len(ans) > 550 else ''))
    print()

# -- Chunk coverage matrix --------------------------------------------------
print('-' * 65)
print('CHUNK COVERAGE -- which strategy found each ensemble chunk')
print('-' * 65)
print(f'  {"#":<3}  {"Naive":<7}  {"Hybrid":<7}  {"HyDE":<7}  {"Agree":<7}  Source / preview')
print('  ' + '-' * 75)

naive_set  = {idx for idx, _ in naive_raw[:TOP_N]}
hybrid_set = {idx for idx, _ in hybrid_raw[:TOP_N]}
hyde_set   = {idx for idx, _ in hyde_raw[:TOP_N]}

for i, r in enumerate(ensemble_chunks, 1):
    idx = r['chunk_idx']
    n_m = 'Y' if idx in naive_set  else '-'
    h_m = 'Y' if idx in hybrid_set else '-'
    y_m = 'Y' if idx in hyde_set   else '-'
    agree   = sum([idx in naive_set, idx in hybrid_set, idx in hyde_set])
    src     = r['source'][:26]
    preview = r['text'][:28].replace('\n', ' ')
    print(f'  {i:<3}  {n_m:<7}  {h_m:<7}  {y_m:<7}  {agree}/3     [{src}] {preview!r}...')

# -- Coverage summary -------------------------------------------------------
print()
ensemble_set        = {r['chunk_idx'] for r in ensemble_chunks}
in_all_three        = ensemble_set & naive_set & hybrid_set & hyde_set
naive_only_contrib  = ensemble_set & naive_set  - hybrid_set - hyde_set
hybrid_only_contrib = ensemble_set & hybrid_set - naive_set  - hyde_set
hyde_only_contrib   = ensemble_set & hyde_set   - naive_set  - hybrid_set

print('Strategy-unique contributions to the ensemble top-5:')
print(f'  All 3 agree:   {len(in_all_three)} chunk(s)  -- strongest cross-modal signal')
print(f'  Naive only:    {len(naive_only_contrib)} chunk(s)  -- semantic match not in keyword/HyDE top-N')
print(f'  Hybrid only:   {len(hybrid_only_contrib)} chunk(s)  -- keyword+semantic combo uniquely surfaced')
print(f'  HyDE only:     {len(hyde_only_contrib)} chunk(s)  -- vocabulary bridging revealed this chunk')
print()
print('-' * 65)
print('ENSEMBLE ANSWER (weighted RRF merged top-5 -- from Cell 4)')
print('-' * 65)
print(answer[:550] + ('...' if len(answer) > 550 else ''))
print()
print('The ensemble answer can cite multiple regulatory sources because the merged')
print('chunk set draws from all four documents; individual strategies may under-retrieve')
print('documents whose vocabulary does not align with their retrieval mechanism.')

## Cell 6: Fintech -- High-Stakes Compliance Answer Verification
**What this demonstrates**: Ensemble RAG applied to a multi-regulator compliance question where missing any one regulatory source is a compliance failure, not a retrieval gap. Shows multi-source chunk attribution, a structured compliance answer citing each source, and why no single-strategy retriever is sufficient for this query type.
**Expected output**: Ensemble chunks with multi-document attribution, a compliance briefing citing each regulatory source, a verification summary with source coverage, and an explanation of why ensemble retrieval is required for this class of query.

In [ ]:
# -- High-stakes compliance query -------------------------------------------
# A compliance officer asking about capital requirements needs:
#   1. Basel III regulatory minimums (CET1, LCR, leverage ratio)
#   2. Bank-specific policy overlays from the lending policy
#   3. Contractual commitments in ISDA collateral schedules
#   4. Disclosed ratios from the earnings report (actual vs. regulatory floor)
# Missing any one source is a compliance failure, not a retrieval failure.
COMPLIANCE_QUERY = (
    'What capital buffers, liquidity requirements, and financial covenants '
    'must the institution maintain to remain in regulatory compliance?'
)

COMPLIANCE_SYSTEM = (
    'You are a Chief Compliance Officer preparing a regulatory capital briefing. '
    'Answer using ONLY the provided document excerpts. '
    'For each requirement, state: (1) the source document, (2) the metric name, '
    '(3) the numeric threshold or ratio. '
    'Organise your answer by type: capital ratios, liquidity ratios, covenants. '
    'If a source document is silent on a requirement type, say so explicitly. '
    'Flag any requirements where thresholds differ across sources.'
)

print('Compliance query:')
print(f'  {COMPLIANCE_QUERY!r}')
print()
print('Running ensemble retrieval with top_k=7 for broader coverage...')

t0 = time.perf_counter()
# top_k=7 instead of 5: compliance answers benefit from broader source coverage
compliance_chunks, comp_lat = ensemble_retrieve(COMPLIANCE_QUERY, n=TOP_N, top_k=7)
retrieval_ms = (time.perf_counter() - t0) * 1000

# -- Print chunks with full source attribution ------------------------------
print(f'Ensemble top-{len(compliance_chunks)} chunks:')
print()
sources_seen: list[str] = []
for i, r in enumerate(compliance_chunks, 1):
    consensus = len(r['found_in'])
    badge     = '***' if consensus == 3 else '** ' if consensus == 2 else '*  '
    via       = '+'.join(r['found_in'])
    src       = r['source']
    if src not in sources_seen:
        sources_seen.append(src)
    preview = r['text'][:95].replace('\n', ' ')
    print(f'  [{i}] {badge} RRF={r["rrf_score"]:.5f}  via [{via}]')
    print(f'       Document: {src}')
    print(f'       {preview!r}...')
    print()

print(f'Source documents represented: {len(sources_seen)} of {len(DOCS)}')
for src in sources_seen:
    n_chunks = sum(1 for r in compliance_chunks if r['source'] == src)
    print(f'  {n_chunks} chunk(s)  <-  {src}')
missing = [d for d in DOCS if d not in sources_seen]
if missing:
    print(f'  Not represented: {missing}')
    print('  -> Consider increasing top_k or adding a domain-specific retriever.')

# -- Generate the compliance briefing ---------------------------------------
print()
print('Generating compliance briefing...')
t_gen = time.perf_counter()
compliance_answer: str = generate_answer(COMPLIANCE_QUERY, compliance_chunks, COMPLIANCE_SYSTEM)
gen_ms = (time.perf_counter() - t_gen) * 1000

print()
print('=' * 65)
print('REGULATORY COMPLIANCE BRIEFING')
print('=' * 65)
print(compliance_answer)
print('=' * 65)

# -- Verification summary ---------------------------------------------------
print()
print('Ensemble verification summary:')
print('  Every claim in the briefing traces to a numbered chunk above.')
print('  A compliance team can verify each claim by reading the source document.')
print()
print('Why each retrieval strategy contributed:')
print('  Naive (dense):   surfaces conceptual capital adequacy passages even when')
print('                   exact regulatory terms differ between query and document.')
print('  Hybrid (BM25+d): ensures exact identifiers (CET1, LCR, NSFR, leverage')
print('                   ratio) are not missed by semantic drift.')
print('  HyDE:            bridges the query abstract compliance language and the')
print("                   earnings report's financial reporting prose register.")
print()
print('A single-strategy retriever would under-retrieve the earnings report narrative')
print('sections and the ISDA contractual covenants. The ensemble captures all sources.')
print()
print('Latency:')
for name in STRATEGY_NAMES:
    print(f'  {name:6s}: {comp_lat.get(name, 0):5.0f} ms')
print(f'  Retrieval wall time: {retrieval_ms:.0f} ms')
print(f'  Answer generation:   {gen_ms:.0f} ms')
print(f'  Total:               {retrieval_ms + gen_ms:.0f} ms')